# Person 4 — Decision Tree Classifier & Model Selection Pipeline

Pipeline Responsibility: Cross-Validation & Model Selection  
Model Assignment: Decision Tree  

This notebook loads the full labelled dataset from `data/raw`, extracts features, tunes `max_depth`, and writes results to `outputs/`.


In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODEL_FOLDER = "decision_tree"
HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in (HERE, *HERE.parents)
        if (p / "data" / "raw").is_dir() and (p / "Basil_Leaf_ML_Workflow.ipynb").exists()
    ),
    None,
)
if ROOT is None:
    ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root containing data/raw.")

MODEL_DIR = ROOT / "parts" / MODEL_FOLDER
OUTPUT_DIR = MODEL_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "parts"))
from _pipeline import CLASSES, FEATURE_VERSION, SEED, prepare_dataset  # noqa: E402

print("Python:", sys.executable)
print("Project root:", ROOT)
print("Model folder:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR)

data = prepare_dataset(ROOT)
X_tr, y_tr = data["X_tr"], data["y_tr"]
X_te, y_te = data["X_te"], data["y_te"]
manifest = data["manifest"]
print(f"Unique images: {len(manifest)} | train: {len(X_tr)} | test: {len(X_te)}")
print("Class counts:", data["audit"]["class_counts"])
print("Dataset complete listed counts:", not data["audit"]["download_coverage"]["partial_dataset"])
print("Feature version:", FEATURE_VERSION)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("--- Person 4: Decision Tree ---")
best_dt, best_score = None, -1.0
for d in (3, 5, 7, 10, None):
    dt = DecisionTreeClassifier(max_depth=d, criterion="entropy", random_state=SEED)
    dt.fit(X_tr, y_tr)
    score = f1_score(y_te, dt.predict(X_te), average="macro", zero_division=0)
    print(f"max_depth={d}: Macro F1 = {score:.4f}")
    if score > best_score:
        best_score, best_dt = score, dt

start_time = time.perf_counter()
best_dt.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time
preds = best_dt.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average="macro", zero_division=0)
cm = confusion_matrix(y_te, preds, labels=CLASSES)
top_feat = np.argsort(best_dt.feature_importances_)[-10:][::-1]
print(f"Best max_depth={best_dt.max_depth} | Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print("Top features:", top_feat)
print("Confusion Matrix:\n", cm)

metrics = {
    "model_name": f"Decision Tree (max_depth={best_dt.max_depth})",
    "pipeline_stage": "Model Selection & Cross Validation",
    "max_depth": best_dt.max_depth,
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "top_10_features": top_feat.tolist(),
    "n_train": int(len(X_tr)),
    "n_test": int(len(X_te)),
    "classes": CLASSES,
    "feature_version": FEATURE_VERSION,
}
(OUTPUT_DIR / "decision_tree_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
joblib.dump(best_dt, OUTPUT_DIR / "decision_tree_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)
